This section borrows heavily from chapter 13, Wheeled Mobile Robots, of Kevin Lynch and Frank Park, *Modern Robotics: Mechanics, Planning, and Control* (Cambridge University Press, 2017) [@Kevin-M-Lynch2017-mp]. You can [download the chapter as a PDF](https://artifacts.aegeanai.com/pdf/lynch-mr/wheeled-robots.pdf).

A wheeled robot's chassis is a planar rigid body, so everything here is built on the planar cases of [rotation representations](/aiml-common/lectures/motion-algebras/rotation-representations/index) and [motion representations](/aiml-common/lectures/motion-algebras/motion-representations/index). You derive the kinematic model of each kind of robot: which wheel speeds produce a desired chassis motion, and which chassis motions are possible at all. Each model is then checked against a physics simulation in which the wheels roll on a floor with friction. Where the model and the simulation agree, the assumptions behind the model hold. Where they disagree, you see exactly which assumption failed, and that is also the reason wheel odometry drifts.

## Wheels and robot types

A *conventional wheel* rolls in its driving direction and does not slide sideways. An *omniwheel* has small passive rollers around its rim, with axes in the plane of the wheel, so it can also slide freely sideways. A *mecanum wheel* is similar, but its rollers are mounted at an angle, typically $45°$, so the free sliding direction makes an angle $\gamma$ with the driving direction.

This difference splits wheeled robots into two families:

- **Omnidirectional robots** use three or more omniwheels or mecanum wheels. Their chassis can move in any direction and rotate at the same time, so any body twist is possible at any instant.
- **Nonholonomic robots** use conventional wheels, as in a differential-drive robot or a car. At each instant the wheels forbid sideways motion, so only a two-dimensional set of body twists is possible, even though the robot can still reach any configuration by maneuvering.

## The chassis as a planar rigid body

The chassis frame $\{b\}$ sits at configuration $q = (\theta, x, y)$ in the space frame $\{s\}$. Its velocity is the body twist $\mathcal{V}_b = (\omega_{bz}, v_{bx}, v_{by})$: the angular velocity, and the linear velocity of the origin of $\{b\}$ written in $\{b\}$. As shown in [twists in the plane](/aiml-common/lectures/motion-algebras/motion-representations/index#twists-in-the-plane), the body twist and the configuration rate are related by

$$
\mathcal{V}_b = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\theta & \sin\theta \\ 0 & -\sin\theta & \cos\theta \end{bmatrix} \dot q .
$$

Both the kinematic models and the simulations below are written in terms of $\mathcal{V}_b$. The simulations record the chassis pose $(\theta, x, y)$ at each frame, differentiate it numerically, and apply this matrix, so the measured twist uses exactly the same definition as the model.

## Omnidirectional robots

Place wheel $i$ at $(x_i, y_i)$ in $\{b\}$, with its driving direction at angle $\beta_i$ from $\hat{x}_b$, radius $r_i$, and free sliding direction at angle $\gamma_i$ from the driving direction. The chassis motion gives the wheel center a linear velocity. Only the component along the driving direction must be produced by the motor, because the rollers absorb the rest. Working this out gives the driving angular speed $u_i$ of wheel $i$ as a linear function of the body twist:

$$
u_i = h_i(0)\, \mathcal{V}_b, \qquad
h_i(0) = \frac{1}{r_i \cos\gamma_i} \begin{bmatrix} x_i \sin(\beta_i + \gamma_i) - y_i \cos(\beta_i + \gamma_i) & \cos(\beta_i + \gamma_i) & \sin(\beta_i + \gamma_i) \end{bmatrix}
$$

Stacking one row per wheel gives $u = H(0)\,\mathcal{V}_b$. When $H(0)$ has rank 3, every body twist has a matching set of wheel speeds, which is what makes the robot omnidirectional. With more than three wheels, $H(0)$ has more rows than columns, and wheel speeds that are not of the form $H(0)\mathcal{V}_b$ make the wheels fight each other and skid.

The code below builds $H(0)$ from the wheel geometry for the two standard designs: three omniwheels spaced $120°$ apart at distance $d$ from the center, and four mecanum wheels at $(\pm\ell, \pm w)$.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rerun as rr
import rerun.blueprint as rrb
import warp as wp

wp.config.log_level = wp.LOG_WARNING
import newton
import newton.viewer
from newton.examples.robot.example_robot_omniwheel import add_mecanum_wheel

RECORDINGS = Path("recordings")
RECORDINGS.mkdir(exist_ok=True)


def wheel_row(x, y, beta, gamma, r):
    # h_i(0): maps the body twist (omega_bz, v_bx, v_by) to the driving speed of one wheel
    a = beta + gamma
    return np.array([x * np.sin(a) - y * np.cos(a), np.cos(a), np.sin(a)]) / (r * np.cos(gamma))


def H_three_omni(d, r):
    # wheels at 90, -30 and 210 degrees, each driving tangentially
    rows = []
    for angle in np.deg2rad([90.0, -30.0, 210.0]):
        rows.append(wheel_row(d * np.cos(angle), d * np.sin(angle), angle - np.pi / 2, 0.0, r))
    return np.array(rows)


def H_mecanum(l, w, r):
    # wheels numbered front-left, front-right, rear-right, rear-left; rollers alternate at -45 and +45 degrees
    specs = [(l, w, -np.pi / 4), (l, -w, np.pi / 4), (-l, -w, -np.pi / 4), (-l, w, np.pi / 4)]
    return np.array([wheel_row(x, y, 0.0, g, r) for x, y, g in specs])


d, r = 0.2, 0.05
H3 = H_three_omni(d, r)
print("three omniwheels, r * H(0):\n", np.round(r * H3, 3))
print("rank", np.linalg.matrix_rank(H3))

l, w = 0.32, 0.40
H4 = H_mecanum(l, w, r)
print("\nfour mecanum wheels, r * H(0):\n", np.round(r * H4, 3))
print("rank", np.linalg.matrix_rank(H4))

Scaled by $r$, the second matrix is Lynch and Park's equation (13.10), with rows $(-\ell - w, 1, -1)$, $(\ell + w, 1, 1)$, $(\ell + w, 1, -1)$ and $(-\ell - w, 1, 1)$. Reading its columns:

- To move along $+\hat{x}_b$, all four wheels drive forward at the same speed.
- To move along $+\hat{y}_b$, wheels 1 and 3 drive backward and wheels 2 and 4 drive forward.
- To rotate counterclockwise, wheels 1 and 4 drive backward and wheels 2 and 3 drive forward.

The chassis can move sideways as fast as it moves forward, because the second and third columns have the same magnitude.

## Simulating the robots

The simulations use Newton, a physics engine for robotics built on NVIDIA Warp [@UnknownUnknown-uf]. A robot is assembled from rigid links connected by joints, with collision shapes that carry a friction coefficient $\mu$. Each wheel is a link on a revolute joint whose motor tracks a commanded angular speed, and the floor is a plane. Nothing in the simulation knows about $H(0)$ or the no-slip condition. The chassis moves only because friction at each contact pushes on the wheels.

The helpers below run a simulation at 60 frames per second, command the wheel speeds at each frame, and record the chassis pose $(\theta, x, y)$ and the wheel angles. The body twist is measured from the recorded poses exactly as defined above. Each run can also be recorded for the Rerun viewer, which draws the robot and the logged paths and signals on one timeline that you can scrub.

In [ ]:
def planar_pose(body_q):
    # (theta, x, y) of a body from its Newton transform (px, py, pz, qx, qy, qz, qw)
    px, py, _, qx, qy, qz, qw = body_q
    theta = math.atan2(2 * (qw * qz + qx * qy), 1 - 2 * (qy * qy + qz * qz))
    return theta, px, py


def measured_body_twist(t, pose):
    # finite-difference the pose, then map q_dot to V_b = (omega_bz, v_bx, v_by)
    theta = np.unwrap(pose[:, 0])
    th_dot, x_dot, y_dot = (np.gradient(v, t) for v in (theta, pose[:, 1], pose[:, 2]))
    c, s = np.cos(theta), np.sin(theta)
    return np.stack([th_dot, c * x_dot + s * y_dot, -s * x_dot + c * y_dot], axis=1)


def simulate(model, chassis, drives, wheel_speeds, duration, recording=None, on_frame=None, blueprint=None,
             fps=60, substeps=6):
    solver = newton.solvers.SolverMuJoCo(
        model, use_mujoco_contacts=False, disable_sensors=True, solver="newton", integrator="implicitfast",
        cone="elliptic", iterations=20, ls_iterations=50, njmax=2048, nconmax=1024,
    )
    state, next_state, control = model.state(), model.state(), model.control()
    newton.eval_fk(model, model.joint_q, model.joint_qd, state)
    collision = newton.CollisionPipeline(model, rigid_contact_max=1024)
    contacts = collision.contacts()
    drive_dofs = [int(model.joint_qd_start.numpy()[j]) for j in drives]
    drive_coords = [int(model.joint_q_start.numpy()[j]) for j in drives]
    target = control.joint_target_qd.numpy()
    dt = 1.0 / fps / substeps

    def advance():
        nonlocal state, next_state
        for _ in range(substeps):
            state.clear_forces()
            collision.collide(state, contacts)
            solver.step(state, next_state, control, contacts, dt)
            state, next_state = next_state, state

    graph = None
    if wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            advance()
        graph = capture.graph

    viewer = None
    if recording is not None:
        viewer = newton.viewer.ViewerRerun(app_id="wheeled_robots_" + recording.stem.split("-0")[0].replace("-", "_"),
                                           record_to_rrd=str(recording), keep_historical_data=True)
        viewer.set_model(model)
        if blueprint is not None:
            rr.send_blueprint(blueprint)

    t, times, poses, angles = 0.0, [], [], []
    for _ in range(int(duration * fps)):
        target[drive_dofs] = wheel_speeds(t)
        control.joint_target_qd.assign(target)
        wp.capture_launch(graph) if graph else advance()
        t += 1.0 / fps
        times.append(t)
        poses.append(planar_pose(state.body_q.numpy()[chassis]))
        angles.append(state.joint_q.numpy()[drive_coords])
        if viewer is not None:
            viewer.begin_frame(t)
            viewer.log_state(state)
            if on_frame is not None:
                on_frame(np.array(times), np.array(poses), np.array(angles))
            viewer.end_frame()
    if viewer is not None:
        viewer.close()
    return np.array(times), np.array(poses), np.array(angles)


def displace(q, twist, dt):
    # move the chassis at a constant body twist for time dt (exponential of the planar twist)
    w, vx, vy = np.asarray(twist) * dt
    if abs(w) < 1e-12:
        db = np.array([0.0, vx, vy])
    else:
        db = np.array([w, (vx * np.sin(w) + vy * (np.cos(w) - 1)) / w, (vy * np.sin(w) + vx * (1 - np.cos(w))) / w])
    c, s = np.cos(q[0]), np.sin(q[0])
    return q + np.array([db[0], c * db[1] - s * db[2], s * db[1] + c * db[2]])


def integrate_twists(q0, twists, dt):
    path = [np.asarray(q0, float)]
    for tw in twists[1:]:
        path.append(displace(path[-1], tw, dt))
    return np.array(path)


VIEWER = "https://artifacts.aegeanai.com/rerun/viewer/0.41.0/index.html?url="
RECORDING_URL = "https://artifacts.aegeanai.com/rerun/wheeled-robots/{}-0.41.rrd"


def show_recording(name, height=560):
    # the published recording in the self-hosted Rerun web viewer
    from urllib.parse import quote
    from IPython.display import IFrame
    src = VIEWER + quote(RECORDING_URL.format(name), safe="")
    return IFrame(src, width="100%", height=height, extras=['allow="fullscreen"', 'loading="lazy"'])


def path_strip(poses, z=0.01):
    return np.column_stack([poses[:, 1], poses[:, 2], np.full(len(poses), z)])


def layout(*plots):
    # the robot in 3D on the left, one time-series panel per signal on the right
    return rrb.Blueprint(
        rrb.Horizontal(rrb.Spatial3DView(origin="/"), rrb.Vertical(*[rrb.TimeSeriesView(origin=f"/plots/{p}", name=p) for p in plots])),
        rrb.BlueprintPanel(state="collapsed"),
        rrb.SelectionPanel(state="collapsed"),
        rrb.TimePanel(state="expanded"),
    )

## The mecanum robot in simulation

The simulated robot has four mecanum wheels, each with twelve passive rollers, at $\ell = 0.32$ m and $w = 0.40$ m. The chassis is commanded through four phases, each lasting three seconds: forward, sideways, turning in place, and a combination of all three. At every frame the wheel speeds come from $u = H(0)\,\mathcal{V}_b$ with the commanded twist.

In [ ]:
MEC_L, MEC_W = 0.32, 0.40


def build_mecanum():
    builder = newton.ModelBuilder()
    builder.default_joint_cfg.damping = 0.01
    chassis = builder.add_link(xform=wp.transform(p=wp.vec3(0.0, 0.0, 0.247), q=wp.quat_identity()), label="chassis")
    builder.add_shape_box(chassis, hx=0.42, hy=0.30, hz=0.06,
                          cfg=newton.ModelBuilder.ShapeConfig(density=80.0, mu=0.8, gap=0.01))
    joints, drives = [builder.add_joint_free(chassis)], []
    # Lynch and Park numbering: front-left, front-right, rear-right, rear-left
    for label, x, y, handedness in [("fl", MEC_L, MEC_W, -1.0), ("fr", MEC_L, -MEC_W, 1.0),
                                    ("rr", -MEC_L, -MEC_W, -1.0), ("rl", -MEC_L, MEC_W, 1.0)]:
        drive, rollers, radius = add_mecanum_wheel(builder, chassis, wp.vec3(x, y, -0.04), handedness, label)
        drives.append(drive)
        joints += [drive, *rollers]
    builder.add_articulation(joints)
    builder.add_ground_plane(cfg=newton.ModelBuilder.ShapeConfig(mu=1.5, gap=0.01))
    return builder.finalize(), chassis, drives, radius


phases = [(0.0, 0.5, 0.0), (0.0, 0.0, 0.5), (0.6, 0.0, 0.0), (0.3, 0.4, 0.2)]  # (omega_bz, v_bx, v_by)


def commanded_twist(t):
    phase = min(int(t // 3.0), len(phases) - 1)
    ramp = min((t - 3.0 * phase) / 0.5, 1.0)  # ease into each phase
    return ramp * np.array(phases[phase])


mec_model, mec_chassis, mec_drives, mec_radius = build_mecanum()
H_sim = H_mecanum(MEC_L, MEC_W, mec_radius)

In [ ]:
def mecanum_frame(t, pose, angles):
    # log the commanded and measured twist and the chassis path so far
    for i, name in enumerate(["omega_bz", "v_bx", "v_by"]):
        rr.log(f"plots/{name}/commanded", rr.Scalars(commanded_twist(t[-1])[i]))
        if len(t) > 2:
            rr.log(f"plots/{name}/measured", rr.Scalars(measured_body_twist(t[-3:], pose[-3:])[-1, i]))
    rr.log("paths/chassis", rr.LineStrips3D([path_strip(pose, z=0.02)], colors=[[255, 170, 0]], radii=0.01))


t_mec, pose_mec, _ = simulate(mec_model, mec_chassis, mec_drives, lambda t: H_sim @ commanded_twist(t), duration=12.0,
                              recording=RECORDINGS / "mecanum-0.41.rrd", on_frame=mecanum_frame,
                              blueprint=layout("omega_bz", "v_bx", "v_by"))

V_meas = measured_body_twist(t_mec, pose_mec)
V_cmd = np.array([commanded_twist(t) for t in t_mec])
for i, (name, _) in enumerate(zip(["forward", "sideways", "rotate", "combined"], phases)):
    window = (t_mec > 3.0 * i + 1.5) & (t_mec < 3.0 * (i + 1) - 0.1)
    print(f"{name:9s} commanded {np.round(V_cmd[window].mean(0), 3)}  measured {np.round(V_meas[window].mean(0), 3)}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, dpi=150)
labels = [r"$\omega_{bz}$ (rad/s)", r"$v_{bx}$ (m/s)", r"$v_{by}$ (m/s)"]
for i, ax in enumerate(axes):
    ax.plot(t_mec, V_cmd[:, i], color="black", linestyle="--", label="commanded")
    ax.plot(t_mec, V_meas[:, i], color="tab:orange", label="measured in simulation")
    ax.set_ylabel(labels[i])
    for edge in (3.0, 6.0, 9.0):
        ax.axvline(edge, color="gray", linewidth=0.5)
axes[0].legend(loc="upper left")
axes[-1].set_xlabel("time (s)")
axes[0].set_title("Mecanum robot: commanded and measured body twist")
plt.tight_layout()
plt.show()

The measured twist follows the command in all four phases, including pure sideways motion, which a robot with conventional wheels cannot produce. The small differences come from the rollers: they are separate rigid bodies that touch the floor one or two at a time, so the effective rolling radius varies slightly as each wheel turns.

In [ ]:
show_recording("mecanum")

*The mecanum robot driven by $u = H(0)\mathcal{V}_b$. Scrub the timeline to follow the four phases; the right-hand panels show the commanded and measured components of the body twist.*

## Nonholonomic robots

A conventional wheel cannot slide sideways. For a robot whose chassis frame sits at the midpoint of the axle between two conventional wheels, this means the chassis origin never moves along $\hat{y}_b$:

$$
v_{by} = -\sin\theta\, \dot x + \cos\theta\, \dot y = 0 .
$$

This is a *Pfaffian constraint* $A(q)\dot q = 0$ with $A(q) = \begin{bmatrix} 0 & -\sin\theta & \cos\theta \end{bmatrix}$. It restricts velocities, not configurations: it cannot be integrated into an equation relating $\theta$, $x$ and $y$ alone. That is what *nonholonomic* means. At any instant the body twist is limited to $\mathcal{V}_b = (\omega, v, 0)$, yet the robot can reach any configuration $(\theta, x, y)$ by maneuvering, the way a driver parallel parks.

The three standard models share this constraint:

- **Unicycle.** A single upright wheel steered about its vertical axis. It is the simplest model with inputs $\omega$ and $v$, and the other two reduce to it.
- **Differential drive.** Two independently driven wheels of radius $r$ at distance $d$ on either side of the chassis origin, plus a passive caster for balance. Turning the left wheel at $u_L$ and the right wheel at $u_R$ gives
  $$
  \omega = \frac{r}{2d}(u_R - u_L), \qquad v = \frac{r}{2}(u_L + u_R),
  $$
  and inverting gives the wheel speeds for a desired $(\omega, v)$: $u_L = (v - d\,\omega)/r$ and $u_R = (v + d\,\omega)/r$.
- **Car.** A car with its chassis frame at the midpoint of the rear axle, wheelbase $\ell$ and steering angle $\psi$ turns at $\omega = v \tan\psi / \ell$. Because $|\psi|$ is bounded, the car has a minimum turning radius $\ell / \tan\psi_{\max}$, which the differential-drive robot does not.

The simulation below checks the differential-drive model. The robot drives straight, turns left, then turns right, first on a floor with friction coefficient $\mu = 1$, then on a floor with $\mu = 0.05$, like ice. Each turn is a half circle of radius 0.6 m at 1 m/s, which needs a sideways acceleration of about $1.7\ \text{m/s}^2$. On the grippy floor friction can supply up to $\mu g \approx 9.8\ \text{m/s}^2$, but on ice only about $0.5\ \text{m/s}^2$.

In [ ]:
DD_R, DD_D = 0.08, 0.18  # wheel radius r and half the distance between the wheels d


def build_diff_drive(mu):
    builder = newton.ModelBuilder()
    builder.default_joint_cfg.damping = 0.01
    # the chassis frame sits at the midpoint of the axle; the body extends backward to a ball caster
    chassis = builder.add_link(xform=wp.transform(p=wp.vec3(0.0, 0.0, DD_R), q=wp.quat_identity()), label="chassis")
    slick = newton.ModelBuilder.ShapeConfig(density=300.0, mu=1e-3, gap=0.01)
    builder.add_shape_box(chassis, xform=wp.transform(p=wp.vec3(-0.06, 0.0, 0.0), q=wp.quat_identity()),
                          hx=0.20, hy=0.13, hz=0.04, cfg=slick)
    builder.add_shape_sphere(chassis, xform=wp.transform(p=wp.vec3(-0.20, 0.0, -(DD_R - 0.03)), q=wp.quat_identity()),
                             radius=0.03, cfg=slick)
    joints, drives = [builder.add_joint_free(chassis)], []
    tire = newton.ModelBuilder.ShapeConfig(density=500.0, mu=mu, gap=0.01)
    axle_to_y = wp.quat_from_axis_angle(wp.vec3(1.0, 0.0, 0.0), -0.5 * wp.pi)
    for name, y in (("left", DD_D), ("right", -DD_D)):
        wheel = builder.add_link(label=name)
        builder.add_shape_cylinder(wheel, xform=wp.transform(q=axle_to_y), radius=DD_R, half_height=0.02, cfg=tire)
        drive = builder.add_joint_revolute(
            parent=chassis, child=wheel, parent_xform=wp.transform(p=wp.vec3(0.0, y, 0.0), q=wp.quat_identity()),
            child_xform=wp.transform(), axis=newton.Axis.Y, target_kd=40.0, damping=0.05, armature=0.02,
            effort_limit=80.0, actuator_mode=newton.JointTargetMode.VELOCITY, label=f"{name}_drive",
        )
        drives.append(drive)
        joints.append(drive)
    builder.add_articulation(joints)
    builder.add_ground_plane(cfg=newton.ModelBuilder.ShapeConfig(mu=mu, gap=0.01))
    return builder.finalize(), chassis, drives


def diff_drive_speeds(omega, v):
    return np.array([(v - DD_D * omega) / DD_R, (v + DD_D * omega) / DD_R])  # (u_L, u_R)


TURN_R, TURN_T = 0.6, np.pi * 0.6 / 1.0  # radius of each turn, and the time a half circle takes at 1 m/s
S_TIMES = (2.0, 2.0 + TURN_T, 2.0 + 2 * TURN_T)  # left turn starts, right turn starts, straight again


def s_curve(t):
    # (omega, v): straight, a half circle to the left, a half circle to the right, straight
    v = min(t / 1.0, 1.0)
    ease = lambda s: np.clip(s / 0.15, 0.0, 1.0)  # enter each turn over 0.15 s
    if S_TIMES[0] <= t < S_TIMES[1]:
        return v / TURN_R * ease(t - S_TIMES[0]), v
    if S_TIMES[1] <= t < S_TIMES[2]:
        return -v / TURN_R * ease(t - S_TIMES[1]), v
    return 0.0, v


T_S = S_TIMES[2] + 1.5
DT = 1.0 / 60
t_grid = np.arange(1, int(T_S * 60) + 1) * DT
model_path = integrate_twists([0.0, 0.0, 0.0], [(w, v, 0.0) for w, v in map(s_curve, t_grid)], DT)

t_dd, pose_grip, _ = simulate(*build_diff_drive(mu=1.0), lambda t: diff_drive_speeds(*s_curve(t)), duration=T_S)

In [ ]:
def ice_frame(t, pose, angles):
    v_by = measured_body_twist(t[-3:], pose[-3:])[-1, 2] if len(t) > 2 else 0.0
    rr.log("plots/v_by", rr.Scalars(v_by))
    rr.log("paths/model", rr.LineStrips3D([path_strip(model_path)], colors=[[70, 130, 255]], radii=0.008), static=True)
    rr.log("paths/grip", rr.LineStrips3D([path_strip(pose_grip)], colors=[[60, 200, 90]], radii=0.008), static=True)
    rr.log("paths/ice", rr.LineStrips3D([path_strip(pose, z=0.02)], colors=[[255, 170, 0]], radii=0.01))


_, pose_ice, _ = simulate(*build_diff_drive(mu=0.05), lambda t: diff_drive_speeds(*s_curve(t)), duration=T_S,
                          recording=RECORDINGS / "diff-drive-0.41.rrd", on_frame=ice_frame, blueprint=layout("v_by"))

vby_grip = measured_body_twist(t_dd, pose_grip)[:, 2]
vby_ice = measured_body_twist(t_dd, pose_ice)[:, 2]
for name, pose, vby in (("mu = 1   ", pose_grip, vby_grip), ("mu = 0.05", pose_ice, vby_ice)):
    gap = np.hypot(pose[-1, 1] - model_path[-1, 1], pose[-1, 2] - model_path[-1, 2])
    print(f"{name}: max |v_by| = {np.abs(vby).max():.3f} m/s, final distance from the model's path = {gap:.2f} m")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
ax1.plot(model_path[:, 1], model_path[:, 2], color="tab:blue", linestyle="--", label="kinematic model")
ax1.plot(pose_grip[:, 1], pose_grip[:, 2], color="tab:green", label=r"simulation, $\mu = 1$")
ax1.plot(pose_ice[:, 1], pose_ice[:, 2], color="tab:orange", label=r"simulation, $\mu = 0.05$")
ax1.set_aspect("equal")
ax1.set_xlabel("x (m)")
ax1.set_ylabel("y (m)")
ax1.set_title("Chassis path")
ax1.legend(loc="best", fontsize=8)
ax2.plot(t_dd, vby_grip, color="tab:green", label=r"$\mu = 1$")
ax2.plot(t_dd, vby_ice, color="tab:orange", label=r"$\mu = 0.05$")
for edge in S_TIMES:
    ax2.axvline(edge, color="gray", linewidth=0.5)
ax2.set_xlabel("time (s)")
ax2.set_ylabel(r"$v_{by}$ (m/s)")
ax2.set_title("Sideways velocity of the chassis origin")
ax2.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

On the grippy floor $v_{by}$ stays near zero through both turns: friction enforces the no-slip constraint that the model assumes, and the simulated path has the model's shape. The turns come out slightly tighter than the model's, because the effective wheel radius and track width differ a little from their nominal values, which the odometry section measures. On ice, friction cannot supply the sideways force the turns need, so the chassis skids outward, $v_{by}$ grows, and the robot ends up far from where the model predicts. The model is not wrong about wheels. Its assumption that wheels do not slip no longer holds.

In [ ]:
show_recording("diff-drive")

*The differential-drive robot on ice. The blue line is the kinematic model's path and the green line the path on a grippy floor; the orange line traces the robot as it skids. The right-hand panel shows $v_{by}$.*

## Odometry

*Odometry* estimates the chassis configuration by integrating wheel motion, which every wheeled robot can measure with encoders. Between two readings, let $\Delta\varphi_L$ and $\Delta\varphi_R$ be the changes in the wheel angles. Assuming the wheels turned at constant speed over the interval, the chassis moved at a constant body twist

$$
\mathcal{V}_b = F \begin{bmatrix} \Delta\varphi_L \\ \Delta\varphi_R \end{bmatrix}, \qquad
F = r \begin{bmatrix} -1/(2d) & 1/(2d) \\ 1/2 & 1/2 \\ 0 & 0 \end{bmatrix},
$$

per unit time. Integrating a constant twist for one unit of time gives the displacement in $\{b\}$ exactly, through the matrix exponential. Rotating that displacement by the current heading and adding it to the current estimate gives the next one. The `displace` function above does exactly this, and the odometry estimate below reuses it.

Odometry errors accumulate, because each step adds its error to all the previous ones. Some errors are systematic, such as a wheel radius or track width that differs from the nominal value, and calibration can remove them. Others, such as slip, are not visible to the encoders at all.

The experiment below first calibrates the effective wheel radius from a straight run and the effective half track $d$ from a steady circle, both at the speed of the test drive. A wheel pressed into the floor rolls on a slightly smaller radius than its nominal one, and it touches the floor across the width of its tread, so neither value matches the drawing exactly. It then drives a long, winding path and compares three odometry estimates with the true path: nominal parameters on a grippy floor, calibrated parameters on the same floor, and calibrated parameters on ice.

For each odometry update, you apply the discrete integration $\mathcal{X}_k = \mathcal{X}_{k-1} \oplus \tau_k$ on SE(2), as described in [calculus on Lie groups](/aiml-common/lectures/motion-algebras/lie-calculus/index).

In [ ]:
def odometry(q0, wheel_angles, r, d):
    F = r * np.array([[-1 / (2 * d), 1 / (2 * d)], [0.5, 0.5], [0.0, 0.0]])
    path = [np.asarray(q0, float)]
    for delta in np.diff(wheel_angles, axis=0):
        path.append(displace(path[-1], F @ delta, 1.0))
    return np.array(path)


# calibration runs at the speed of the test drive: straight ahead, then a steady circle
_, pose_s, ang_s = simulate(*build_diff_drive(mu=1.0), lambda t: diff_drive_speeds(0.0, 0.6 * min(t, 1.0)), duration=6.0)
k = len(pose_s) // 3  # skip the start, where the robot is still speeding up
r_eff = np.hypot(pose_s[-1, 1] - pose_s[k, 1], pose_s[-1, 2] - pose_s[k, 2]) / np.mean(ang_s[-1] - ang_s[k])
_, pose_c, ang_c = simulate(*build_diff_drive(mu=1.0), lambda t: diff_drive_speeds(0.6 * min(t, 1.0), 0.6 * min(t, 1.0)),
                            duration=8.0)
k = len(pose_c) // 4
turned = np.unwrap(pose_c[:, 0])[-1] - np.unwrap(pose_c[:, 0])[k]
d_eff = r_eff * ((ang_c[-1, 1] - ang_c[k, 1]) - (ang_c[-1, 0] - ang_c[k, 0])) / (2 * turned)
print(f"nominal    r = {DD_R:.4f} m, d = {DD_D:.4f} m")
print(f"calibrated r = {r_eff:.4f} m, d = {d_eff:.4f} m")


def winding(t):
    v = 0.6 * min(t / 1.0, 1.0)
    return 0.9 * np.sin(2 * np.pi * t / 12.0), v

In [ ]:
def odometry_frame(t, pose, angles):
    # the calibrated estimate, advanced by the latest wheel-angle increment
    F = r_eff * np.array([[-1 / (2 * d_eff), 1 / (2 * d_eff)], [0.5, 0.5], [0.0, 0.0]])
    estimate = pose[0] if len(t) == 1 else displace(live[-1], F @ (angles[-1] - angles[-2]), 1.0)
    live.append(estimate)
    rr.log("paths/true", rr.LineStrips3D([path_strip(pose, z=0.02)], colors=[[255, 170, 0]], radii=0.01))
    rr.log("paths/odometry", rr.LineStrips3D([path_strip(np.array(live))], colors=[[70, 130, 255]], radii=0.008))
    rr.log("plots/position_error", rr.Scalars(float(np.hypot(*(estimate[1:] - pose[-1, 1:])))))


T_ODO = 36.0
live = []
t_o, pose_true, ang_true = simulate(*build_diff_drive(mu=1.0), lambda t: diff_drive_speeds(*winding(t)), duration=T_ODO,
                                    recording=RECORDINGS / "odometry-0.41.rrd", on_frame=odometry_frame,
                                    blueprint=layout("position_error"))

_, pose_ice_o, ang_ice_o = simulate(*build_diff_drive(mu=0.05), lambda t: diff_drive_speeds(*winding(t)), duration=T_ODO)

estimates = {
    "nominal r and d, grippy floor": (pose_true, odometry(pose_true[0], ang_true, DD_R, DD_D)),
    "calibrated r and d, grippy floor": (pose_true, odometry(pose_true[0], ang_true, r_eff, d_eff)),
    "calibrated r and d, ice": (pose_ice_o, odometry(pose_ice_o[0], ang_ice_o, r_eff, d_eff)),
}
errors = {k: np.hypot(est[:, 1] - true[:, 1], est[:, 2] - true[:, 2]) for k, (true, est) in estimates.items()}
for k, e in errors.items():
    print(f"{k:34s} error after {T_ODO:.0f} s: {e[-1]:.3f} m")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), dpi=150, gridspec_kw={"width_ratios": [1, 1.6]})
colors = ["tab:red", "tab:blue", "tab:purple"]
ax1.plot(pose_true[:, 1], pose_true[:, 2], color="black", linewidth=2, label="true path, grippy floor")
for (k, (_, est)), c in zip(estimates.items(), colors[:2]):
    ax1.plot(est[:, 1], est[:, 2], color=c, linestyle="--", label=f"odometry, {k.split(',')[0]}")
ax1.set_aspect("equal")
ax1.set_xlabel("x (m)")
ax1.set_ylabel("y (m)")
ax1.set_title("True path and odometry on the grippy floor")
ax1.legend(loc="best", fontsize=8)
for (k, e), c in zip(errors.items(), colors):
    ax2.plot(t_o, e, color=c, label=k)
ax2.set_xlabel("time (s)")
ax2.set_ylabel("position error (m)")
ax2.set_title("Odometry error over time")
ax2.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

With nominal parameters the error grows steadily, because every wheel revolution is converted to distance with a slightly wrong radius and every turn to heading with a slightly wrong track width. Calibration reduces this systematic error substantially but does not remove it, because no single pair of values for $r$ and $d$ matches every part of a winding drive. On ice the calibrated estimate fails again: the wheels turn, but the chassis does not move the way they report, and no choice of $r$ and $d$ can account for that. This is why robots combine odometry, which is accurate over short times, with sensors whose errors do not accumulate, such as range sensors, cameras or GPS, inside an estimator such as a [Kalman filter](/aiml-common/lectures/rse/kalman-filters/index). The [state estimation and mapping](/book/state-estimation-mapping/index) chapter develops that machinery.

In [ ]:
show_recording("odometry")

*The winding drive on the grippy floor. The orange line is the true path and the blue line the calibrated odometry estimate; the right-hand panel shows their distance over time.*

## Further topics

Chapter 13 of Lynch and Park goes on to the *controllability* of nonholonomic robots (why a car can parallel park even though it cannot move sideways), *motion planning* under nonholonomic constraints, including shortest paths for a car with a minimum turning radius, *feedback control* for tracking a reference trajectory, and *mobile manipulation*, where an arm is mounted on a wheeled base. The planning lectures in this course take up path planning for mobile robots.